In [1]:
!pip install sentence-transformers
!pip install pdfplumber
!pip install scikit-learn
!pip install joblib

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.6/43.6 kB 2.4 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 68.1/68.1 kB 3.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.0/60.0 kB 2.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.6/6.6 MB 79.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 59.1 MB/s eta 0:00:00


In [ ]:
import os
import pdfplumber
import joblib
import numpy as np
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity

In [12]:
model = SentenceTransformer("all-MiniLM-L6-v2")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [13]:
def extract_text_from_pdf(file_path):

    text = ""

    with pdfplumber.open(file_path) as pdf:

        for page in pdf.pages:

            page_text = page.extract_text()

            if page_text:
                text += page_text

    return text

In [14]:
class ResumeATS:

    def __init__(self, model):

        self.model = model

    def compute_embedding(self, text):

        return self.model.encode([text])


    def rank_resumes(self, jd_text, resumes):

        """
        resumes = list of resume texts
        """

        jd_embedding = self.model.encode([jd_text])

        resume_embeddings = self.model.encode(resumes)

        scores = cosine_similarity(jd_embedding, resume_embeddings)[0]

        ranking = []

        for i, score in enumerate(scores):

            ranking.append({
                "resume_id": i,
                "score": float(score)
            })

        ranking = sorted(ranking, key=lambda x: x["score"], reverse=True)

        return ranking

In [15]:
ats_system = ResumeATS(model)

In [16]:
jd_text = """
Looking for a backend developer with experience in Python,
FastAPI, SQL, AWS, and Docker.
"""

In [17]:
resume1 = """
Python developer with experience in FastAPI and SQL databases.
Worked on REST APIs and cloud deployment.
"""

resume2 = """
Frontend developer skilled in React, JavaScript and CSS.
"""

resume3 = """
Machine learning engineer with Python, TensorFlow, and AWS.
"""

resumes = [resume1, resume2, resume3]

In [18]:
results = ats_system.rank_resumes(jd_text, resumes)

print(results)

[{'resume_id': 0, 'score': 0.8170028328895569}, {'resume_id': 2, 'score': 0.5924544334411621}, {'resume_id': 1, 'score': 0.49782687425613403}]


In [19]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [24]:
jd_path = "/content/Aquera - JD for Backend Developer (4) (2).pdf"
resume_path = "/content/RESUME (Aquera).pdf"

In [25]:
jd_text = extract_text_from_pdf(jd_path)

resume_text = extract_text_from_pdf(resume_path)

print(jd_text[:500])      # preview
print(resume_text[:500])

Title: Backend Developer
Our technology stack is carefully designed to be reactive (responsive, elastic, resilient, message
driven), so if you want to participate in building class products, and be part of a vibrant and world
class tech team then we have a lot to offer you.
We are looking for developers at various levels starting at 2 - 4 years and another one at 7 - 10
years of total experience with a minimum of 2 years and 4 years of relevant experience in writing
ES6 (javascript) script and N
Rishi Raj
+91 7321807126 | rishirajskpuram@gmail.com | LinkedIn | GitHub | Leetcode
Education
Nitte Meenakshi Institute of Technology, Bengaluru, Karnataka Expected 2026
B.E. in Information Science and Engineering CGPA: 8.22
Technical Skills
Languages: Java, Python, JavaScript, HTML, CSS
Frameworks / Libraries: MERN Stack (MongoDB, Express.js, React, Node.js)
Tools / Platforms: Git, GitHub, VS Code, Postman
Cloud / Databases: AWS (Fundamentals), MySQL, MongoDB
Relevant Coursework: Data Structur

In [26]:
jd_embedding = model.encode([jd_text])

resume_embedding = model.encode([resume_text])

In [27]:
score = cosine_similarity(jd_embedding, resume_embedding)[0][0]

print("Match Score:", round(score*100,2), "%")

Match Score: 40.69 %


In [29]:
import os

resume_folder = "/content/resumes"

resumes = []
names = []

for file in os.listdir(resume_folder):

    if file.endswith(".pdf"):

        path = os.path.join(resume_folder, file)

        text = extract_text_from_pdf(path)

        resumes.append(text)
        names.append(file)

In [30]:
resume_embeddings = model.encode(resumes)

scores = cosine_similarity(jd_embedding, resume_embeddings)[0]

In [31]:
import pandas as pd

results = pd.DataFrame({
    "Resume": names,
    "Score": scores
})

results["Score"] = (results["Score"]*100).round(2)

results = results.sort_values(by="Score", ascending=False)

results

,Resume,Score
4,RESUME (W2025).pdf,52.360001
0,RESUME (fs).pdf,43.369999
1,RESUME.pdf,40.689999
3,RESUME (gen).pdf,40.689999
5,RESUME (Aquera).pdf,40.689999
2,RESUME (ml).pdf .pdf,22.040001


In [32]:
top = results.iloc[0]

print("Best Candidate:", top["Resume"])
print("ATS Score:", top["Score"], "%")

Best Candidate: RESUME (W2025).pdf
ATS Score: 52.36 %
